In [1]:
import pandas as pd

In [2]:
food_df = pd.read_csv("/content/food.csv")
food_nutrient_df = pd.read_csv("/content/food_nutrient.csv")
nutrient_df = pd.read_csv("/content/nutrient.csv")
foundation_food_df = pd.read_csv("/content/foundation_food.csv")

/tmp/ipykernel_3460/2813590165.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  food_nutrient_df = pd.read_csv("/content/food_nutrient.csv")


In [3]:
food_df.head(2)

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
1,319875,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01


In [4]:
food_nutrient_df.head(2)

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired
0,2201847,319877,1051,56.30,1.0,1.0,NaN,NaN,NaN,NaN,NaN
1,2201845,319877,1002,1.28,1.0,1.0,NaN,NaN,NaN,NaN,NaN


In [5]:
nutrient_df.head(2)

,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.0,280.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0


In [6]:
foundation_food_df.head()

,fdc_id,NDB_number,footnote
0,321358,16158,NaN
1,321360,100147,NaN
2,321611,11056,NaN
3,323121,7022,NaN
4,323294,12563,Other phytosterols = 34.67 mg/100g


In [ ]:
import requests

def get_meals_by_macros(min_protein=40, max_fat=100, number_of_recipes=3):
    
    api_key = "448533b1425842039d118d4f6529ad6f"
    url = "https://api.spoonacular.com/recipes/findByNutrients"

    params = {
        "minProtein": min_protein,
        "maxFat": max_fat,
        "number": number_of_recipes,
        "apiKey": api_key
    }

    try:
        response = requests.get(url, params=params)
        if response.status_code == 200:
            recipes = response.json()

            cleaned_results = []
            for recipe in recipes:
                cleaned_results.append({
                    "Title": recipe["title"],
                    "Calories": recipe["calories"],
                    "Protein": recipe["protein"],
                    "Fat": recipe["fat"],
                    "Carbs": recipe["carbs"],
                    "Image": recipe["image"]
                })
            return cleaned_results
        else:
            return f"Error: {response.status_code} - {response.text}"

    except Exception as e:
        return f"An error occurred: {str(e)}"

suggested_meals = get_meals_by_macros(min_protein=40, max_fat=30)
print(suggested_meals)

[{'Title': 'Summertime Seafood Pie', 'Calories': 572, 'Protein': '69g', 'Fat': '18g', 'Carbs': '0g', 'Image': 'https://img.spoonacular.com/recipes/662271-312x231.jpg'}, {'Title': 'Slow Cooker Chili', 'Calories': 452, 'Protein': '50g', 'Fat': '11g', 'Carbs': '0g', 'Image': 'https://img.spoonacular.com/recipes/673457-312x231.jpg'}, {'Title': 'Instant Pot Pressure Cooker Pot Roast', 'Calories': 484, 'Protein': '51g', 'Fat': '9g', 'Carbs': '0g', 'Image': 'https://img.spoonacular.com/recipes/982375-312x231.jpg'}]


In [ ]:
foundation_ids = foundation_food_df['fdc_id'].unique()
food_filtered = food_df[food_df['fdc_id'].isin(foundation_ids)]


# 1008 = Energy (Calories), 1003 = Protein, 1004 = Total lipid (Fat), 1005 = Carbohydrate
macro_ids = [1008, 1003, 1004, 1005]
food_nutrient_filtered = food_nutrient_df[
    (food_nutrient_df['fdc_id'].isin(foundation_ids)) &
    (food_nutrient_df['nutrient_id'].isin(macro_ids))
]

merged_df = pd.merge(
    food_nutrient_filtered[['fdc_id', 'nutrient_id', 'amount']],
    food_filtered[['fdc_id', 'description']],
    on='fdc_id',
    how='inner'
)


pivot_df = merged_df.pivot_table(
    index=['fdc_id', 'description'],
    columns='nutrient_id',
    values='amount'
).reset_index()

# 1008 -> Calories, 1003 -> Protein, 1004 -> Fat, 1005 -> Carbs
pivot_df = pivot_df.rename(columns={
    1008: 'Calories_per_100g',
    1003: 'Protein_per_100g',
    1004: 'Fat_per_100g',
    1005: 'Carbs_per_100g'
})

pivot_df = pivot_df.fillna(0)

pivot_df.to_csv('egyptian_agents_food_db.csv', index=False)



✅ اللعبة اتقفلت بنجاح! تم إنشاء ملف: egyptian_agents_food_db.csv
📊 إجمالي المكونات الخام الجاهزة للاستخدام: 355 مكون.


In [8]:
pivot_df

nutrient_id,fdc_id,description,Protein_per_100g,Fat_per_100g,Carbs_per_100g,Calories_per_100g
0,321358,"Hummus, commercial",7.350000,17.1000,14.900000,229.0
1,321360,"Tomatoes, grape, raw",0.830000,0.6300,5.510000,27.0
2,321611,"Beans, snap, green, canned, regular pack, drai...",1.040000,0.3900,4.110000,21.0
3,323121,"Frankfurter, beef, unheated",11.700000,28.0000,2.890000,314.0
4,323294,"Nuts, almonds, dry roasted, with salt added",20.400000,57.8000,16.200000,620.0
...,...,...,...,...,...,...
350,2747673,"Tuna, ahi or yellowfin, frozen, wild caught",24.700000,0.3875,-0.104500,0.0
351,2747674,"Turnips, raw",0.953125,0.1188,7.274975,0.0
352,2747675,"Watermelon, seedless, flesh only, raw",0.871250,0.0000,0.000000,0.0
353,2747676,"Watermelon, seedless, rind only, raw",0.531250,0.0700,4.166000,0.0


In [11]:
pivot_df["description"].unique()

array(['Hummus, commercial', 'Tomatoes, grape, raw',
       'Beans, snap, green, canned, regular pack, drained solids',
       'Frankfurter, beef, unheated',
       'Nuts, almonds, dry roasted, with salt added', 'Kale, raw',
       'Egg, whole, raw, frozen, pasteurized',
       'Egg, white, raw, frozen, pasteurized', 'Egg, white, dried',
       'Onion rings, breaded, par fried, frozen, prepared, heated in oven',
       'Pickles, cucumber, dill or kosher dill',
       'Cheese, parmesan, grated',
       'Cheese, pasteurized process, American, vitamin D fortified',
       'Grapefruit juice, white, canned or bottled, unsweetened',
       'Peaches, yellow, raw',
       'Seeds, sunflower seed kernels, dry roasted, with salt added',
       'Kale, frozen, cooked, boiled, drained, without salt',
       'Mustard, prepared, yellow', 'Kiwifruit, green, raw',
       'Nectarines, raw', 'Cheese, cheddar',
       'Cheese, cottage, lowfat, 2% milkfat',
       'Cheese, mozzarella, low moisture, part-ski

In [14]:
def search_food_macros(food_name):
    """أداة للـ Agent عشان يبحث في قاعدة البيانات المحلية عن ماكروز أي أكلة"""
    df = pd.read_csv('/content/egyptian_agents_food_db.csv')

    # البحث بالاسم (case-insensitive)
    results = df[df['description'].str.contains(food_name, case=False, na=False)]

    output = []
    for _, row in results.head(5).iterrows(): # هيرجع أول 5 نتائج مطابقة بس
        output.append({
            "Food": row['description'],
            "Calories_100g": row['Calories_per_100g'],
            "Protein_100g": row['Protein_per_100g'],
            "Fat_100g": row['Fat_per_100g'],
            "Carbs_100g": row['Carbs_per_100g']
        })
    return output

# تجربة الأدآة:
print(search_food_macros("Cottage"))

[{'Food': 'Cheese, cottage, lowfat, 2% milkfat', 'Calories_100g': 84.0, 'Protein_100g': 11.0, 'Fat_100g': 2.3, 'Carbs_100g': 4.31}, {'Food': 'Cottage cheese, full fat, large or small curd', 'Calories_100g': 0.0, 'Protein_100g': 11.62436, 'Fat_100g': 4.225, 'Carbs_100g': 4.59964}]
